### Skapar dataset med och utan de kategoriska features

In [44]:
import pandas as pd
df = pd.read_csv('revised_data.csv', index_col=0)
df = df.drop(columns='id') # Tar bort id eftersom id = index, onödig feature
df['gender'] = df['gender'].apply(lambda x: 'Woman' if x == 1 else 'Man') # Eftersom det är en kategorisk variabel gör jag den mer läsbar
yes_cat = df.drop(columns=[ 'ap_hi', 'ap_lo', 'height', 'weight', 'BMI'])
no_cat = df.drop(columns=['BMI_class', 'blood_pressure', 'height', 'weight'])

#One hot encoding
yes_cat = pd.get_dummies(yes_cat, columns=['BMI_class', 'blood_pressure', 'gender'])
no_cat = pd.get_dummies(no_cat, columns=['gender'])


### Använder custom class för att ta fram modeller att testa datan

In [45]:
from model_class import model_selection
#Initierar två klasser med och utan kategoriska features
X = yes_cat.drop(columns='cardio')
y = yes_cat['cardio']
#Initierar med skalering och test_split funktion
with_cat = model_selection(X, y, autoscale=True, autosplit=True)
X = no_cat.drop(columns='cardio')
y = no_cat['cardio']
#Initierar med skalering och test_split funktion
without_cat = model_selection(X, y, autoscale=True, autosplit=True)


## Val av modeller
##### Eftersom detta är ett kvalificeringsproblem väljer jag följande:
- KNN
- Logistisk Regression
- Decision Trees
- Ridge Classifier
- RandomForestClassifier

In [46]:
#Skapar Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#Ihopklistrad och omskriven kod. Ursprungligen från ChatGpt och sklearn
#Definiera Parametrar med respektive modell
param_grid = [
    {
        'classifier': [KNeighborsClassifier()],
        'classifier__n_neighbors': [1,3],
    },
    {
        'classifier': [LogisticRegression()],
        'classifier__C': [0.01, 0.1, 1]
    },
    {
        'classifier': [DecisionTreeClassifier()],
        'classifier__max_depth': [1,2]
    },
    {
        'classifier': [RidgeClassifier()],
        'classifier__alpha': [0.01, 0.1, 1]
    },
    {
        'classifier': [RandomForestClassifier()],
        'classifier__max_depth': [1, 2],
    }
]
print('With Categories:')
with_cat.GridCV_pipeline_fit(param_grid=param_grid)
print('Without Categories:')
without_cat.GridCV_pipeline_fit(param_grid=param_grid)
#Tar ca 50 sekunder, RandomForest är långsam på cpu

With Categories:
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1;, score=0.612 total time=   0.1s
[CV 2/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1;, score=0.605 total time=   0.1s
[CV 3/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1;, score=0.607 total time=   0.1s
[CV 4/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1;, score=0.608 total time=   0.1s
[CV 5/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1;, score=0.597 total time=   0.1s
[CV 1/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=3;, score=0.641 total time=   0.1s
[CV 2/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=3;, score=0.627 total time=   0.1s
[CV 3/5] END classifier=KNeighborsClassifier(), classifier__n_neighbors=3;, score=0.633 total time=   0.1s
[CV 4/5] END classifier=KNeighborsClassifier(), classifier__n_neig

In [62]:
#Hämtar bästa parametrar för varje model
def get_all_best_params(lst_params, lst_scores):
    score_index = 0
    output = list()
    active = lst_params[0]
    for index, param in enumerate(lst_params):
        if active['classifier'] != param['classifier']:
            output.append(f"{active}, score: {lst_scores[score_index]}\n")
            active = param
            score_index = index
        else:
            if lst_scores[score_index] < lst_scores[index]:
                score_index = index
                active = param
    
    output.append(f"{active}, score: {lst_scores[score_index]}\n")
    return ''.join(output)
            
            

score_withcat = get_all_best_params(with_cat.grid_search.cv_results_['params'], with_cat.grid_search.cv_results_['mean_test_score'])
score_withoutcat = get_all_best_params(without_cat.grid_search.cv_results_['params'], without_cat.grid_search.cv_results_['mean_test_score'])

print('Category dataset:')
print(score_withcat)
print('Without category dataset:')
print(score_withoutcat)

Category dataset:
{'classifier': KNeighborsClassifier(), 'classifier__n_neighbors': 3}, score: 0.6338704718160091
{'classifier': LogisticRegression(), 'classifier__C': 0.01}, score: 0.6924434834065338
{'classifier': DecisionTreeClassifier(), 'classifier__max_depth': 2}, score: 0.6760575901460212
{'classifier': RidgeClassifier(), 'classifier__alpha': 1}, score: 0.6920804455041012
{'classifier': RandomForestClassifier(), 'classifier__max_depth': 2}, score: 0.6773161250557804

Without category dataset:
{'classifier': KNeighborsClassifier(), 'classifier__n_neighbors': 3}, score: 0.669135343538575
{'classifier': LogisticRegression(), 'classifier__C': 0.1}, score: 0.7228674397197927
{'classifier': DecisionTreeClassifier(), 'classifier__max_depth': 1}, score: 0.7069655179160322
{'classifier': RidgeClassifier(), 'classifier__alpha': 1}, score: 0.7218750845714793
{'classifier': RandomForestClassifier(), 'classifier__max_depth': 2}, score: 0.7116609264476792

